In [2]:
import pandas as pd
import numpy as np
import datetime
import tempfile
import os
import subprocess
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'code')))
from example_run_jobs import *


In [ ]:
dataset_name = "casp"
dataset_path = "data/%s/%s.csv" % (dataset_name, dataset_name)
full_dataset_path = "%s/%s" % (os.getcwd(), dataset_path)

positions = {"lov": ["G2", "T112"],
             "pard3": ["L48","R82"],
             "gcn4": ["S101","S144"],
             "pte": ["I72", "M283"],
             "his": ["L7", "D211"],
             "aamyl": ["P4", "D425"],
             "casp": ["D561", "R588"]§  }
             
df = pd.read_csv(full_dataset_path)
si = np.where(df.columns == positions[dataset_name][0])[0][0]
ei = np.where(df.columns == positions[dataset_name][1])[0][0]+1 
pos_to_embed = df.columns[si:ei].to_list()


In [4]:
if sum(df["num_muts"] == 0) == 0:
    print("Adding wild type row")
    wt_p_aa = [(int(p[1:]), p[0]) for p in df.columns[si:ei]]

    ref_seqs = []
    random_indexes = np.random.choice(range(0, len(df)), size=30, replace=False)

    for i in random_indexes:
        ref_seq = [c for c in df["full_seq"][0]]

        for p, aa in wt_p_aa:
            ref_seq[p-1] = aa

        ref_seq = "".join(ref_seq)
        ref_seqs.append(ref_seq)


    assert sum([rfs == ref_seqs[0] for rfs in ref_seqs]) == len(ref_seqs)

    wt_row = df.iloc[0].copy()

    wt_row["full_seq"] = ref_seqs[0]
    wt_row["activity"] = 1
    wt_row["binned_activity"] = 1
    wt_row["num_muts"] = 0.0
    wt_row["sanity_mut_numb"] = 0


    for (tuple, pos) in zip(wt_p_aa, df.columns[si:ei]):
        wt_row[pos] = tuple[1]

    print(wt_row)

    fdf = df.append(wt_row, ignore_index=True)
    fdf
    fdf.to_csv(full_dataset_path, index=False)



In [5]:
pos_to_embed_fin = [p[1:] for p in pos_to_embed]

In [6]:
all_models =  ["esm2_t6_8M_UR50D","esm2_t12_35M_UR50D", "esm2_t30_150M_UR50D", "esm2_t33_650M_UR50D",
               "esm2_t36_3B_UR50D","progen2-small", "progen2-medium","prot_bert"]


In [6]:
ix = -1
all_models[ix+1]

'esm2_t6_8M_UR50D'

In [7]:
model_name = "esm2_t36_3B_UR50D"#all_models[]

args = ["../code/embedding_engine.py",
        "--action", "embed_parallel",
        "--dataset_file", "%s" % full_dataset_path,
        #"--n_workers", "5",
        #"--execute_after_parallel_generation", "false",
        "--model", "%s" % model_name, #### ENTER MODEL NAME HERE
        "--evaluation_path", "/%s/results/%s/%s" % (os.getcwd(), dataset_name, model_name),
        "--sequence_colname", "full_seq",
        "--label_column_name", "activity",
        "--job_executing_type", "lsf",
        "--chunk_size", "50",
        "--internal_batch_size", "4",
        "--positions_to_embed"] + pos_to_embed_fin



print(args)

['../code/embedding_engine.py', '--action', 'embed_parallel', '--dataset_file', '/home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/data/casp/casp.csv', '--model', 'esm2_t36_3B_UR50D', '--evaluation_path', '//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t36_3B_UR50D', '--sequence_colname', 'full_seq', '--label_column_name', 'activity', '--job_executing_type', 'lsf', '--chunk_size', '50', '--internal_batch_size', '4', '--positions_to_embed', '561', '562', '563', '565', '566', '567', '568', '571', '572', '573', '574', '575', '576', '577', '578', '579', '580', '581', '582', '583', '584', '585', '586', '587', '588']


In [50]:

out, err = run_python_script(args, cwd=os.path.abspath("../code"), return_both=True)
print(out)
print(err)

Running command: python ../code/embedding_engine.py --action check_on_jobs --jobs_path //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t30_150M_UR50D/jobs_20260420_202522_parallel
[INFO] Initializing PLM with current working directory: /home/labs/fleishman/itayta/new_fitness_repo/fitness_learning
[INFO] Checking on jobs in: //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t30_150M_UR50D/jobs_20260420_202522_parallel
[INFO] Args: {'average_embeddings': False, 'positions_to_embed': [561, 562, 563, 565, 566, 567, 568, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588], 'dataset_file': '/home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/data/casp/casp.csv', 'n_workers': 5, 'chunk_size': 50, 'internal_batch_size': 6, 'print_done_jobs': False, 'execute_after_parallel_generation': False, 'model': 'esm2_t30_150M_UR50D', 'evaluation_path': '//home/labs

In [26]:
out

"[INFO] Initializing PLM with current working directory: /home/labs/fleishman/itayta/new_fitness_repo/fitness_learning\n[INFO] Successfully loaded '/home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/data/his/his.csv' with column 'full_seq'.\n[INFO] Using evaluation_path: //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/esm2_t33_650M_UR50D\n"

In [57]:

JOBS_PATH = ""
jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6_8M_UR50D/jobs_20260420_202708_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t12_35M_UR50D/jobs_20260420_202752_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t30_150M_UR50D/jobs_20260420_202522_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/progen2-small/jobs_20260420_202433_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/progen2-medium/jobs_20260420_202323_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/prot_bert/jobs_20260420_202158_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t33_650M_UR50D/jobs_20260420_202024_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t36_3B_UR50D/jobs_20260420_201857_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/esm2_t6_8M_UR50D/jobs_20260420_112302_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/esm2_t12_35M_UR50D/jobs_20260420_112021_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/prot_bert/jobs_20260420_111857_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/progen2-small/jobs_20260420_111752_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/progen2-medium/jobs_20260420_111618_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/esm2_t30_150M_UR50D/jobs_20260420_111532_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/aamyl/esm2_t33_650M_UR50D/jobs_20260420_111428_parallel"

#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/his/esm2_t6_8M_UR50D/jobs_20260420_111044_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/his/esm2_t12_35M_UR50D/jobs_20260420_110828_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/his/prot_bert/jobs_20260420_105703_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/his/progen2-small/jobs_20260420_105530_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/his/progen2-medium/jobs_20260420_105052_parallel"
#jobs_path = "//home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/his/esm2_t36_3B_UR50D/jobs_20260418_193230_parallel" #out.split("JOBS PATH")[1].split("\n")[0].split(" ")[-1]
args = ["../code/embedding_engine.py",
        "--action", "execute_workers",
        "--job_executing_type", "lsf",
        "--jobs_path", "%s" % jobs_path,
        "--n_workers", "100"]


# In case something fails!
# args = ["../code/embedding_engine.py",
#         "--action", "execute_workers",
#         "--job_executing_type", "lsf",
#         "--jobs_path", "%s" % jobs_path, 
#         "--specific_job_i", "1046",
#         "--n_workers", "10"]

# run_python_script(args, cwd=os.path.abspath("../code")) 

In [58]:

run_python_script(args, cwd=os.path.abspath("../code"))

Running command: python ../code/embedding_engine.py --action execute_workers --job_executing_type lsf --jobs_path //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6_8M_UR50D/jobs_20260420_202708_parallel --n_workers 100
[INFO] Initializing PLM with current working directory: /home/labs/fleishman/itayta/new_fitness_repo/fitness_learning
[INFO] Created job execution library at: //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6_8M_UR50D/jobs_20260420_202708_parallel/job_execution_20260420_211405
Submitting job 1 via:  bsub -n 2 -gpu num=1:gmem=24G:aff=yes -R same[gpumodel] -R rusage[mem=64GB] -R span[ptile=2] -e //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6_8M_UR50D/jobs_20260420_202708_parallel/job_execution_20260420_211405/job_outputs/err_file_worker_44338449 -o //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6

<Popen: returncode: 0 args: ['python', '../code/embedding_engine.py', '--act...>

In [60]:
args = ["../code/embedding_engine.py",
        "--action", "check_on_jobs",
        "--jobs_path", "%s" % jobs_path]

run_python_script(args, cwd=os.path.abspath("../code"))

Running command: python ../code/embedding_engine.py --action check_on_jobs --jobs_path //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6_8M_UR50D/jobs_20260420_202708_parallel
[INFO] Initializing PLM with current working directory: /home/labs/fleishman/itayta/new_fitness_repo/fitness_learning
[INFO] Checking on jobs in: //home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/results/casp/esm2_t6_8M_UR50D/jobs_20260420_202708_parallel
[INFO] Args: {'average_embeddings': False, 'positions_to_embed': [561, 562, 563, 565, 566, 567, 568, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588], 'dataset_file': '/home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/data/casp/casp.csv', 'n_workers': 5, 'chunk_size': 200, 'internal_batch_size': 10, 'print_done_jobs': False, 'execute_after_parallel_generation': False, 'model': 'esm2_t6_8M_UR50D', 'evaluation_path': '//home/labs/fleish

<Popen: returncode: 0 args: ['python', '../code/embedding_engine.py', '--act...>

In [ ]:
#model_to_collect = model_name
hidden_dim_to_model_dict = {
    "prot_bert": 1024,
    "progen2-small": 1024,
    "progen2-medium": 1536,
    "esm2_t6_8M_UR50D": 320,
    "esm2_t12_35M_UR50D": 480,
    "esm2_t30_150M_UR50D": 640,
    "esm2_t33_650M_UR50D": 1280,
    "esm2_t36_3B_UR50D": 2560,
}


dataset_name = "his"

for model_to_collect in ["esm2_t30_650M_UR50D"]:
    model_hidden_dim = hidden_dim_to_model_dict[model_to_collect]
    csv_name = "data/%s/%s.csv" % (dataset_name, dataset_name)
    cmd = "python collect_protgym_embeddings.py --model_name %s --dataset_name %s --hidden_dim_size %d --input_csv %s" % (model_to_collect, dataset_name, model_hidden_dim, csv_name)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    tmp_dir_name = f"collect_all_job_results_{timestamp}"
    tmp_dir_path = os.path.join(os.getcwd(),  "tmp_files", tmp_dir_name)

    os.makedirs(tmp_dir_path, exist_ok=True)

    run_lsf_job(cmd, 
                local_script_dir=tmp_dir_path,
                code_execution_dir=os.getcwd(),
                conda_env_name="esm_env",
                worker_idx=None,
                gpu=False,
                N_cores=6,
                rusage="64GB")

python collect_protgym_embeddings.py --model_name esm2_t33_650M_UR50D --dataset_name his --hidden_dim_size 1280 --input_csv data/his/his.csv


In [ ]:
 #python train_classifiers_over_ohe.py --dataset_name his --regression --external_labels_column activity   --op adam --n_start 1 --n_end 6 --hl 200 --niters 200
  #python train_classifiers_over_ohe.py --dataset_name aamyl --regression --external_labels_column activity   --op lbfgs --n_start 1 --n_end 6 --hl 20 --niters 1000
  #python train_classifiers_over_embeddings.py --model_name esm_150m --dataset_name his  --mean_embeddings --regression --external_labels_column activity --hl 200 200 --op adam --niters 200 --n_start 4 --n_end 6 --min_n 1

  #python train_classifiers_over_ohe.py --dataset_name his --regression --external_labels_column activity   --op adam --n_start 1 --n_end 6 --hl 200 --niters 20